# 03 — Seller Features and Shrinkage (Logistic Regression)

Continues from notebook 02. Precision at the cost-optimal threshold was
still only 0.25. This notebook adds new features to the existing logistic
regression model — most importantly, seller-level historical statistics —
to see whether there's more signal available in the data than the original
feature set captured.


## New feature: seller history

Computed **from training data only**, so no test-set information leaks
into these aggregates:

- `seller_avg_delay` — this seller's average delivery delay across their order history
- `seller_bad_rate` — the share of this seller's past orders that were "bad"
- `seller_order_count` — how much history exists on this seller


In [ ]:
if "seller_id" not in data.columns:
    data = data.merge(items[["order_id", "seller_id"]], on="order_id", how="left")

train_seller_lookup = data.loc[x_train.index, ["seller_id", "delay_days"]].copy()
train_seller_lookup["bad_order"] = y_train

seller_stats = train_seller_lookup.groupby("seller_id").agg(
    seller_avg_delay=("delay_days", "mean"),
    seller_bad_rate=("bad_order", "mean"),
    seller_order_count=("bad_order", "count"),
).reset_index()

global_avg_delay = train_seller_lookup["delay_days"].mean()
global_bad_rate = train_seller_lookup["bad_order"].mean()

def apply_seller_stats(df_slice):
    merged = data.loc[df_slice.index, ["seller_id"]].merge(seller_stats, on="seller_id", how="left")
    merged.index = df_slice.index
    merged["seller_avg_delay"] = merged["seller_avg_delay"].fillna(global_avg_delay)
    merged["seller_bad_rate"] = merged["seller_bad_rate"].fillna(global_bad_rate)
    merged["seller_order_count"] = merged["seller_order_count"].fillna(0)
    return merged[["seller_avg_delay", "seller_bad_rate", "seller_order_count"]]

x_train = pd.concat([x_train, apply_seller_stats(x_train)], axis=1)
x_test = pd.concat([x_test, apply_seller_stats(x_test)], axis=1)

for split_df in [x_train, x_test]:
    split_df["freight_installment_interaction"] = split_df["freight_ratio"] * split_df["payment_installments"]
    split_df["delay_freight_interaction"] = split_df["delay_days"] * split_df["freight_ratio"]

num_features_v2 = num_features + [
    "seller_avg_delay", "seller_bad_rate", "seller_order_count",
    "freight_installment_interaction", "delay_freight_interaction",
]


## Group rare categories

Before fitting, check whether any `product_category_name`/`customer_state` value has too little training volume to produce a stable coefficient.

In [ ]:
CATEGORY_MIN_COUNT = 200
STATE_MIN_COUNT = 150

train_category_counts = x_train["product_category_name"].value_counts()
rare_categories = set(train_category_counts[train_category_counts < CATEGORY_MIN_COUNT].index)

train_state_counts = x_train["customer_state"].value_counts()
rare_states = set(train_state_counts[train_state_counts < STATE_MIN_COUNT].index)

print(f"Grouping {len(rare_categories)} rare product categories into 'Other'")
print(f"Grouping {len(rare_states)} rare customer regions into 'Other'")

for split_df in [x_train, x_test]:
    split_df["product_category_name"] = split_df["product_category_name"].apply(
        lambda v: "Other" if v in rare_categories else v)
    split_df["customer_state"] = split_df["customer_state"].apply(
        lambda v: "Other" if v in rare_states else v)


## Fit logistic regression on the expanded (v2) feature set

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_features_v2),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])
model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0))
])
model.fit(x_train[num_features_v2 + cat_features], y_train)
probs = model.predict_proba(x_test[num_features_v2 + cat_features])[:, 1]

print("AUC-ROC (v2):", roc_auc_score(y_test, probs))
print("AUC-PR (v2):", average_precision_score(y_test, probs))
print(classification_report(y_test, probs > 0.5))


## Coefficient check — instability from small-sample categories

In [ ]:
coef_df = pd.DataFrame({
    "feature": num_features_v2 + list(model.named_steps["prep"].named_transformers_["cat"].get_feature_names_out(cat_features)),
    "coefficient": model.named_steps["clf"].coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)
print(coef_df.head(10))


Even after grouping, some individual category/state coefficients dominated
the top of the list with counts in the low hundreds — e.g. `Party Supplies`
(36 rows), `Manipur` (70 rows). Coefficients estimated from that few
observations, in a ~15% positive-rate problem, are statistically unreliable:
a coefficient of +1.0 from 36 rows is close to noise with a confident number
attached. Rare-category grouping (above) mitigates this; anything still
appearing prominently after grouping should be caveated rather than
presented as a strong business finding.

## IV on the v2 feature set

Re-running WOE/IV (same function as notebook 02) with the new features
included:


In [ ]:
train_data_v2 = x_train.copy()
train_data_v2["bad_order"] = y_train

iv_summary = {}
for feat in num_features_v2:
    _, _, iv = calculate_woe_iv(train_data_v2, feat, "bad_order", bins=5, is_categorical=False)
    iv_summary[feat] = iv
for feat in cat_features:
    _, _, iv = calculate_woe_iv(train_data_v2, feat, "bad_order", is_categorical=True)
    iv_summary[feat] = iv

iv_data = pd.DataFrame.from_dict(iv_summary, orient="index", columns=["IV"]).sort_values("IV", ascending=False)
print(iv_data)


`seller_bad_rate` shows up with IV ≈ 0.42 — the "Strong" band, similar
territory to `delay_days` (0.39). Since `seller_bad_rate` is a **target-encoded**
feature (it's literally the training-set mean of the label, grouped by
seller), high IV here raises a different concern than `delay_days` did:
target encoding is prone to overfitting on sellers with very few orders —
a seller with 1 order that happened to be bad gets `seller_bad_rate = 1.0`,
an overconfident estimate from a single data point.


## Correlation check

Before assuming any coefficient instability is from multicollinearity between the seller features:

In [ ]:
print(x_train[["delay_days", "seller_avg_delay", "seller_bad_rate"]].corr())


Correlations came back moderate (0.24–0.34) — not high enough to explain instability through simple multicollinearity, so any odd coefficient behavior isn't primarily a correlation artifact.

## Sensitivity check: does the model still work without seller features?

In [ ]:
x_train_no_seller = x_train.drop(columns=["seller_bad_rate", "seller_avg_delay"])
x_test_no_seller = x_test.drop(columns=["seller_bad_rate", "seller_avg_delay"])
num_features_no_seller = [f for f in num_features_v2 if f not in ("seller_bad_rate", "seller_avg_delay")]

preprocessor_no_seller = ColumnTransformer([
    ("num", StandardScaler(), num_features_no_seller),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])
model_no_seller = Pipeline([
    ("prep", preprocessor_no_seller),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0))
])
model_no_seller.fit(x_train_no_seller[num_features_no_seller + cat_features], y_train)
probs_no_seller = model_no_seller.predict_proba(x_test_no_seller[num_features_no_seller + cat_features])[:, 1]

print("AUC-ROC without seller features:", roc_auc_score(y_test, probs_no_seller))
print("AUC-PR without seller features:", average_precision_score(y_test, probs_no_seller))


**Result**: AUC-ROC drops from 0.711 to 0.672 without seller features (confirms
real, load-bearing signal), but **AUC-PR actually recovers** to nearly the
v1 baseline (0.422 vs. 0.384 with seller features included, unshrunk). This
is the tell: the unshrunk `seller_bad_rate` improves broad ranking while
*hurting* precision at the sharp end of the distribution — exactly what
you'd expect from unstable, overconfident low-sample seller estimates.
Fix: shrinkage.


## Shrinkage fix

Blends each seller's raw rate toward the global average, weighted by how many orders that seller actually has.

In [ ]:
GLOBAL_BAD_RATE = train_seller_lookup["bad_order"].mean()
SHRINKAGE_WEIGHT = 20  # higher = more shrinkage toward the global average

seller_stats["seller_bad_rate_raw"] = seller_stats["seller_bad_rate"]  # kept for comparison
seller_stats["seller_bad_rate"] = (
    (seller_stats["seller_bad_rate_raw"] * seller_stats["seller_order_count"]) +
    (GLOBAL_BAD_RATE * SHRINKAGE_WEIGHT)
) / (seller_stats["seller_order_count"] + SHRINKAGE_WEIGHT)

x_train = x_train.drop(columns=["seller_avg_delay", "seller_bad_rate", "seller_order_count"], errors="ignore")
x_test = x_test.drop(columns=["seller_avg_delay", "seller_bad_rate", "seller_order_count"], errors="ignore")
x_train = pd.concat([x_train, apply_seller_stats(x_train)], axis=1)
x_test = pd.concat([x_test, apply_seller_stats(x_test)], axis=1)

preprocessor_v2 = ColumnTransformer([
    ("num", StandardScaler(), num_features_v2),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])
model_v2 = Pipeline([
    ("prep", preprocessor_v2),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, C=1.0))
])
model_v2.fit(x_train[num_features_v2 + cat_features], y_train)
probs_v2_shrunk = model_v2.predict_proba(x_test[num_features_v2 + cat_features])[:, 1]

print("AUC-ROC (shrunk seller features):", roc_auc_score(y_test, probs_v2_shrunk))
print("AUC-PR (shrunk seller features):", average_precision_score(y_test, probs_v2_shrunk))
print(classification_report(y_test, probs_v2_shrunk > 0.5))

cost_fp, cost_fn = 50, 300
thresholds = np.arange(0.1, 0.9, 0.01)
best_thresh, best_cost = None, float("inf")
for t in thresholds:
    preds = (probs_v2_shrunk > t).astype(int)
    fp = ((preds == 1) & (y_test == 0)).sum()
    fn = ((preds == 0) & (y_test == 1)).sum()
    total_cost = fp * cost_fp + fn * cost_fn
    if total_cost < best_cost:
        best_cost, best_thresh = total_cost, t

print(f"Optimal threshold: {best_thresh}, cost: {best_cost}")
print(classification_report(y_test, probs_v2_shrunk > best_thresh))


## Result

| Version | AUC-ROC | AUC-PR | Cost |
|---|---|---|---|
| v1 baseline | 0.673 | 0.422 | ₹747,350 |
| v2, unshrunk seller_bad_rate | 0.711 | 0.384 | ₹686,500 |
| v2, no seller features | 0.672 | 0.422 | — |
| **v2, shrunk seller_bad_rate** | 0.709 | **0.408** | ₹689,200 |

Shrinkage recovered most of the AUC-PR the unshrunk version was losing,
while keeping nearly all of the AUC-ROC gain. The shrunk version's cost is
marginally higher than the unshrunk one (₹689,200 vs. ₹686,500 — a 0.4%
difference, within noise), but it's the more honest model: the unshrunk
version's slightly better cost was partly an artifact of overconfident
predictions on low-volume sellers, which wouldn't generalize as reliably
to genuinely new sellers in production.

Precision at this point is still capped in the low 0.30s by logistic
regression itself — the model can't represent non-linear thresholds or
interaction effects it wasn't explicitly given. Notebook 04 tries XGBoost
on this same feature set.
